# Question Classification with Sentence Embeddings

**Client:** Chat Plus  
**Goal:** Predict question type labels using sentence embeddings, then compare performance across two embedding models.

This notebook is structured to match the project requirements:

1. Process the data  
2. Run supervised classification  
3. Evaluate performance and form a hypothesis  
4. Test the hypothesis with additional data from the holdout set  
5. Compare final performance after retraining

## Step 0: Setup

This notebook assumes you have `train.csv` and `test.csv` uploaded into your Colab session.  
The dataset columns are:

- `label-coarse`
- `label-fine`
- `text`

We will use these two embedding models:

- `all-MiniLM-L6-v2`
- `paraphrase-MiniLM-L6-v2`

We use **logistic regression** as the supervised **linear** classifier.

In [5]:
!pip install -q sentence-transformers scikit-learn pandas numpy matplotlib seaborn


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [6]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    log_loss
)

plt.rcParams["figure.figsize"] = (10, 6)
sns.set_context("talk")

## Step 1: Load and inspect the data

In [7]:
train_df = pd.read_csv('question_classificatrion/question_classification_dataset/train.csv')
test_df = pd.read_csv('question_classificatrion/question_classification_dataset/test.csv')

print("Provided train shape:", train_df.shape)
print("Provided test shape:", test_df.shape)
display(train_df.head())

print("\nNumber of coarse labels:", train_df["label-coarse"].nunique())
print("Number of fine labels:", train_df["label-fine"].nunique())

Provided train shape: (5452, 3)
Provided test shape: (500, 3)


,label-coarse,label-fine,text
0,0,0,How did serfdom develop in and then leave Russ...
1,1,1,What films featured the character Popeye Doyle ?
2,0,0,How can I find a list of celebrities ' real na...
3,1,2,What fowl grabs the spotlight after the Chines...
4,2,3,What is the full form of .com ?



Number of coarse labels: 6
Number of fine labels: 47


## Step 2: Build one combined dataset, then create a fresh random split

### Why recombine the provided train/test files?
The project instructions ask for a **random train/test/holdout split**, so we will combine the provided files and create our own split.

### Tradeoffs of different split choices
**50/25/25**  
- Pros: more training data than a 25/25/50 split  
- Cons: smaller holdout set for final hypothesis testing

**25/25/50**  
- Pros: very strong final holdout set for later testing  
- Cons: much less training data, which can hurt model quality

**60/20/20 or 70/15/15**  
- Pros: often best for maximizing training performance  
- Cons: less room for a true final evaluation and manual hypothesis testing

### Team choice used in this notebook
We use **25% train / 25% test / 50% holdout**, because the assignment explicitly emphasizes:
- forming a hypothesis,
- manually sourcing new data from the holdout set,
- and measuring performance change after adding new examples.

We will use **stratified splitting by fine label** so the class distribution stays more balanced across splits.

In [8]:
full_df = pd.concat([train_df, test_df], ignore_index=True)
print("Combined shape:", full_df.shape)

# Optional: focus on 8 fine-grained categories so analysis is more manageable.
# We choose the 8 most frequent fine labels to reduce extreme class imbalance.
top_k = 8
top_fine_labels = full_df["label-fine"].value_counts().head(top_k).index.tolist()

focused_df = full_df[full_df["label-fine"].isin(top_fine_labels)].copy()
focused_df = focused_df.reset_index(drop=True)

print("Focused dataset shape:", focused_df.shape)
print("Selected fine labels:", top_fine_labels)
display(focused_df["label-fine"].value_counts().sort_index())

Combined shape: (5952, 3)
Focused dataset shape: (3821, 3)
Selected fine labels: [4, 14, 7, 13, 12, 0, 8, 1]


label-fine
0      278
1      207
4     1017
7      544
8      265
12     331
13     372
14     807
Name: count, dtype: int64

### Why these 8 categories?
We use the **8 most frequent fine labels** because:
- they give enough examples per class for training and evaluation,
- they reduce noise from extremely small categories,
- and they keep the presentation more focused.

You can change `top_k` or manually select different labels later if your team wants a different subset.

In [9]:
# First split off the 50% holdout set
train_test_df, holdout_df = train_test_split(
    focused_df,
    test_size=0.50,
    random_state=42,
    stratify=focused_df["label-fine"]
)

# Split the remaining half into 25% train / 25% test overall
train_split_df, test_split_df = train_test_split(
    train_test_df,
    test_size=0.50,
    random_state=42,
    stratify=train_test_df["label-fine"]
)

print("Train shape:", train_split_df.shape)
print("Test shape:", test_split_df.shape)
print("Holdout shape:", holdout_df.shape)

print("\nTrain fine-label distribution:")
display(train_split_df["label-fine"].value_counts().sort_index())

print("\nTest fine-label distribution:")
display(test_split_df["label-fine"].value_counts().sort_index())

print("\nHoldout fine-label distribution:")
display(holdout_df["label-fine"].value_counts().sort_index())

Train shape: (955, 3)
Test shape: (955, 3)
Holdout shape: (1911, 3)

Train fine-label distribution:


label-fine
0      69
1      52
4     254
7     136
8      67
12     82
13     93
14    202
Name: count, dtype: int64


Test fine-label distribution:


label-fine
0      70
1      52
4     254
7     136
8      66
12     83
13     93
14    201
Name: count, dtype: int64


Holdout fine-label distribution:


label-fine
0     139
1     103
4     509
7     272
8     132
12    166
13    186
14    404
Name: count, dtype: int64

## Step 3: Choose two embedding models and explain why

We compare:

1. **all-MiniLM-L6-v2**  
   - widely used, lightweight, and fast  
   - strong baseline for semantic similarity and sentence classification

2. **paraphrase-MiniLM-L6-v2**  
   - optimized for paraphrase-style semantic matching  
   - useful for checking whether paraphrase-focused embeddings help question-type classification

Our goal is to compare:
- classification quality,
- confusion patterns,
- and speed / performance tradeoffs.

In [10]:
embedding_models = [
    "all-MiniLM-L6-v2",
    "paraphrase-MiniLM-L6-v2"
]

X_train_text = train_split_df["text"].tolist()
X_test_text = test_split_df["text"].tolist()
X_holdout_text = holdout_df["text"].tolist()

y_train = train_split_df["label-fine"].to_numpy()
y_test = test_split_df["label-fine"].to_numpy()
y_holdout = holdout_df["label-fine"].to_numpy()

class_names = sorted(train_split_df["label-fine"].unique().tolist())
class_names_str = [str(x) for x in class_names]

print("Classes used:", class_names)

Classes used: [0, 1, 4, 7, 8, 12, 13, 14]


## Helper functions

In [11]:
def embed_texts(model_name, texts):
    model = SentenceTransformer(model_name)
    start = time.time()
    embeddings = model.encode(texts, convert_to_numpy=True, show_progress_bar=True)
    elapsed = time.time() - start
    return embeddings, elapsed

def train_and_evaluate_linear_classifier(X_train, y_train, X_eval, y_eval):
    clf = LogisticRegression(max_iter=4000)
    start = time.time()
    clf.fit(X_train, y_train)
    train_time = time.time() - start

    eval_pred = clf.predict(X_eval)
    eval_proba = clf.predict_proba(X_eval)

    metrics = {
        "accuracy": accuracy_score(y_eval, eval_pred),
        "macro_f1": f1_score(y_eval, eval_pred, average="macro"),
        "weighted_f1": f1_score(y_eval, eval_pred, average="weighted"),
        "log_loss": log_loss(y_eval, eval_proba, labels=clf.classes_)
    }
    return clf, eval_pred, eval_proba, metrics, train_time

def evaluate_on_multiple_splits(clf, X_train, y_train, X_test, y_test, X_holdout, y_holdout):
    train_pred = clf.predict(X_train)
    train_proba = clf.predict_proba(X_train)

    test_pred = clf.predict(X_test)
    test_proba = clf.predict_proba(X_test)

    holdout_pred = clf.predict(X_holdout)
    holdout_proba = clf.predict_proba(X_holdout)

    rows = []
    for split_name, y_true, y_pred, y_proba in [
        ("train", y_train, train_pred, train_proba),
        ("test", y_test, test_pred, test_proba),
        ("holdout", y_holdout, holdout_pred, holdout_proba),
    ]:
        rows.append({
            "split": split_name,
            "accuracy": accuracy_score(y_true, y_pred),
            "macro_f1": f1_score(y_true, y_pred, average="macro"),
            "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
            "log_loss": log_loss(y_true, y_proba, labels=clf.classes_)
        })

    return pd.DataFrame(rows), test_pred, holdout_pred

def plot_conf_mat(y_true, y_pred, labels, title):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(title)
    plt.tight_layout()
    plt.show()

def per_class_f1_df(y_true, y_pred, labels, model_name):
    report = classification_report(y_true, y_pred, labels=labels, output_dict=True, zero_division=0)
    rows = []
    for label in labels:
        rows.append({
            "model": model_name,
            "label-fine": label,
            "precision": report[str(label)]["precision"],
            "recall": report[str(label)]["recall"],
            "f1-score": report[str(label)]["f1-score"],
            "support": report[str(label)]["support"],
        })
    return pd.DataFrame(rows)

def plot_metric_comparison(results_df, metric, title):
    pivot = results_df.pivot(index="model", columns="split", values=metric)
    pivot = pivot[["train", "test", "holdout"]]
    pivot.plot(kind="bar")
    plt.title(title)
    plt.ylabel(metric)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()

## Step 4: Embed the data and train the supervised classifier

In [12]:
all_results = []
per_class_results = []
artifacts = {}

for model_name in embedding_models:
    print(f"\n===== Running {model_name} =====")

    X_train_emb, train_embed_time = embed_texts(model_name, X_train_text)
    X_test_emb, test_embed_time = embed_texts(model_name, X_test_text)
    X_holdout_emb, holdout_embed_time = embed_texts(model_name, X_holdout_text)

    clf = LogisticRegression(max_iter=4000)
    start = time.time()
    clf.fit(X_train_emb, y_train)
    fit_time = time.time() - start

    metrics_df, test_pred, holdout_pred = evaluate_on_multiple_splits(
        clf,
        X_train_emb, y_train,
        X_test_emb, y_test,
        X_holdout_emb, y_holdout
    )

    metrics_df["model"] = model_name
    metrics_df["embed_time_train_sec"] = train_embed_time
    metrics_df["embed_time_test_sec"] = test_embed_time
    metrics_df["embed_time_holdout_sec"] = holdout_embed_time
    metrics_df["fit_time_sec"] = fit_time
    all_results.append(metrics_df)

    per_class_df = per_class_f1_df(y_test, test_pred, class_names, model_name)
    per_class_results.append(per_class_df)

    artifacts[model_name] = {
        "X_train_emb": X_train_emb,
        "X_test_emb": X_test_emb,
        "X_holdout_emb": X_holdout_emb,
        "clf": clf,
        "test_pred": test_pred,
        "holdout_pred": holdout_pred
    }

results_df = pd.concat(all_results, ignore_index=True)
per_class_df = pd.concat(per_class_results, ignore_index=True)

display(results_df)


===== Running all-MiniLM-L6-v2 =====


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...

Batches:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/60 [00:00<?, ?it/s]

TypeError: LogisticRegression.__init__() got an unexpected keyword argument 'multi_class'

### Basic performance comparison

In [ ]:
plot_metric_comparison(results_df, "accuracy", "Accuracy by Model and Split")
plot_metric_comparison(results_df, "macro_f1", "Macro F1 by Model and Split")
plot_metric_comparison(results_df, "log_loss", "Log Loss by Model and Split")

## Confusion matrices for each embedding model

In [ ]:
for model_name in embedding_models:
    plot_conf_mat(
        y_test,
        artifacts[model_name]["test_pred"],
        class_names,
        f"Test Confusion Matrix - {model_name}"
    )

## Visualization to isolate where one embedding is better than the other

This compares **per-class F1 score** on the test set.
Positive bars mean the second model is better for that class.
Negative bars mean the first model is better.

In [ ]:
compare_df = per_class_df.pivot(index="label-fine", columns="model", values="f1-score").reset_index()
compare_df["difference"] = compare_df[embedding_models[1]] - compare_df[embedding_models[0]]
display(compare_df)

plt.figure(figsize=(10, 6))
plt.bar(compare_df["label-fine"].astype(str), compare_df["difference"])
plt.axhline(0, linestyle="--")
plt.xlabel("Fine Label")
plt.ylabel(f"F1 Difference: {embedding_models[1]} - {embedding_models[0]}")
plt.title("Per-Class F1 Difference Between Embedding Models")
plt.tight_layout()
plt.show()

## Step 5: Evaluate classification performance

Use the cells below to answer:

- What worked well?
- Which classes performed worst?
- Did the model generalize well?
- How do train vs test vs holdout metrics compare?
- Are there signs of overfitting?
- What types of wording or question structure may be causing confusion?

In [ ]:
# Generalization gap: compare train and test accuracy / macro F1
gen_view = results_df.pivot(index="model", columns="split", values=["accuracy", "macro_f1", "log_loss"])
display(gen_view)

In [ ]:
# Inspect mistakes for each model
for model_name in embedding_models:
    error_df = test_split_df.copy()
    error_df["pred"] = artifacts[model_name]["test_pred"]
    error_df["correct"] = error_df["label-fine"] == error_df["pred"]

    print(f"\nWorst mistakes to inspect for {model_name}")
    display(error_df[~error_df["correct"]][["text", "label-fine", "pred"]].head(20))

### Example team hypothesis template

**Hypothesis:** The model struggles most when two categories use similar question wording, especially short factual questions beginning with words like *what*, *where*, or *who*.  
Because of that, adding more examples from the weakest classes—especially examples with overlapping wording patterns—should improve recall and F1 for those classes.

You should revise this hypothesis after inspecting the confusion matrices and example errors above.

## Step 6: Test the hypothesis by adding new examples from the holdout set

### Strategy in this notebook
We will:
1. identify the weakest classes from the **test** set,
2. manually source more examples from the **holdout** set for those classes,
3. add them to training,
4. retrain both models,
5. compare the new confusion matrices and metrics.

You can change how many examples to add with `examples_per_target_class`.

In [ ]:
# Find weakest classes based on average F1 across both models
avg_class_perf = per_class_df.groupby("label-fine")["f1-score"].mean().sort_values()
display(avg_class_perf)

target_classes = avg_class_perf.head(2).index.tolist()  # weakest 2 classes
examples_per_target_class = 15

print("Target classes for additional data:", target_classes)
print("Examples added per target class:", examples_per_target_class)

In [ ]:
# Manually source extra data from holdout for the target classes
holdout_for_sampling = holdout_df.copy()

sampled_extra_parts = []
for cls in target_classes:
    class_pool = holdout_for_sampling[holdout_for_sampling["label-fine"] == cls]
    sampled = class_pool.sample(
        n=min(examples_per_target_class, len(class_pool)),
        random_state=42
    )
    sampled_extra_parts.append(sampled)

extra_df = pd.concat(sampled_extra_parts, ignore_index=True)

print("Extra examples pulled from holdout:", extra_df.shape)
display(extra_df[["label-fine", "text"]].head(20))

# Remove sampled rows from holdout for a cleaner post-update evaluation
remaining_holdout_df = holdout_df.drop(index=extra_df.index, errors="ignore").copy()

augmented_train_df = pd.concat([train_split_df, extra_df], ignore_index=True)

print("Augmented train shape:", augmented_train_df.shape)
print("Remaining holdout shape:", remaining_holdout_df.shape)

### Why this amount of additional data?
This notebook uses **15 examples per weakest class** as a reasonable starting point:
- large enough to potentially shift the decision boundary,
- small enough to preserve a meaningful remaining holdout set,
- and easy to justify in a presentation.

You can increase or decrease this depending on your holdout size and how strong your hypothesis is.

In [ ]:
X_train_text_aug = augmented_train_df["text"].tolist()
y_train_aug = augmented_train_df["label-fine"].to_numpy()

X_holdout_text_new = remaining_holdout_df["text"].tolist()
y_holdout_new = remaining_holdout_df["label-fine"].to_numpy()

post_results = []
post_per_class = []
post_artifacts = {}

for model_name in embedding_models:
    print(f"\n===== Retraining with added data: {model_name} =====")

    X_train_emb_aug, train_embed_time_aug = embed_texts(model_name, X_train_text_aug)
    X_test_emb, test_embed_time_aug = embed_texts(model_name, X_test_text)
    X_holdout_emb_new, holdout_embed_time_aug = embed_texts(model_name, X_holdout_text_new)

    clf_aug = LogisticRegression(max_iter=4000)
    start = time.time()
    clf_aug.fit(X_train_emb_aug, y_train_aug)
    fit_time_aug = time.time() - start

    metrics_df_aug, test_pred_aug, holdout_pred_aug = evaluate_on_multiple_splits(
        clf_aug,
        X_train_emb_aug, y_train_aug,
        X_test_emb, y_test,
        X_holdout_emb_new, y_holdout_new
    )

    metrics_df_aug["model"] = model_name
    metrics_df_aug["stage"] = "after_augmentation"
    metrics_df_aug["embed_time_train_sec"] = train_embed_time_aug
    metrics_df_aug["embed_time_test_sec"] = test_embed_time_aug
    metrics_df_aug["embed_time_holdout_sec"] = holdout_embed_time_aug
    metrics_df_aug["fit_time_sec"] = fit_time_aug
    post_results.append(metrics_df_aug)

    per_class_aug = per_class_f1_df(y_test, test_pred_aug, class_names, model_name)
    post_per_class.append(per_class_aug)

    post_artifacts[model_name] = {
        "clf": clf_aug,
        "test_pred": test_pred_aug,
        "holdout_pred": holdout_pred_aug
    }

post_results_df = pd.concat(post_results, ignore_index=True)
post_per_class_df = pd.concat(post_per_class, ignore_index=True)

display(post_results_df)

## New confusion matrices after adding data

In [ ]:
for model_name in embedding_models:
    plot_conf_mat(
        y_test,
        post_artifacts[model_name]["test_pred"],
        class_names,
        f"Test Confusion Matrix After Added Data - {model_name}"
    )

## Step 7: Final performance comparison

In [ ]:
baseline_results = results_df.copy()
baseline_results["stage"] = "baseline"

combined_results = pd.concat([baseline_results, post_results_df], ignore_index=True)
display(combined_results)

In [ ]:
for metric in ["accuracy", "macro_f1", "log_loss"]:
    compare_plot_df = combined_results[combined_results["split"].isin(["test", "holdout"])].copy()
    pivot = compare_plot_df.pivot_table(
        index=["model", "stage"],
        columns="split",
        values=metric
    )
    display(pivot)

    pivot.plot(kind="bar")
    plt.title(f"{metric} Before vs After Adding Data")
    plt.ylabel(metric)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
# Per-class change in F1 after adding data
baseline_pc = per_class_df.copy()
baseline_pc["stage"] = "baseline"

post_pc = post_per_class_df.copy()
post_pc["stage"] = "after_augmentation"

pc_compare = pd.concat([baseline_pc, post_pc], ignore_index=True)

for model_name in embedding_models:
    model_pc = pc_compare[pc_compare["model"] == model_name]
    pivot = model_pc.pivot(index="label-fine", columns="stage", values="f1-score").reset_index()
    pivot["change"] = pivot["after_augmentation"] - pivot["baseline"]
    display(pivot)

    plt.figure(figsize=(10, 6))
    plt.bar(pivot["label-fine"].astype(str), pivot["change"])
    plt.axhline(0, linestyle="--")
    plt.xlabel("Fine Label")
    plt.ylabel("F1 Change")
    plt.title(f"Per-Class F1 Change After Adding Data - {model_name}")
    plt.tight_layout()
    plt.show()

## Final discussion prompts

Use your results to answer the following:

### 1. What did and did not work well?
- Which model had better **test** and **holdout** performance?
- Which classes remained difficult?
- Which classes improved most after adding data?

### 2. Did the model generalize well?
A model generalizes better when:
- train and test performance are fairly close,
- holdout performance is similar to test performance,
- and improvements are not limited to only the training set.

### 3. Did the model behave as expected after adding data?
Look at:
- changes in confusion matrices,
- changes in macro F1,
- changes in per-class F1,
- and any drop or increase in holdout performance.

### 4. How did embedding choice affect results?
Compare:
- accuracy / F1,
- confusion patterns,
- embedding time,
- and fit time.

A model can be slightly more accurate but much slower, so you should discuss the **performance versus time payoff**.

## Optional: Create your own challenge examples

Use this section to test your hypothesis with custom questions not in the test set.

In [ ]:
custom_examples = [
    "Who invented the telescope?",
    "What is the capital of Peru?",
    "When was the internet invented?",
    "Where is the Sahara Desert located?"
]

def predict_custom_examples(model_name, texts):
    model = SentenceTransformer(model_name)
    X_custom = model.encode(texts, convert_to_numpy=True, show_progress_bar=False)
    clf = post_artifacts[model_name]["clf"]
    preds = clf.predict(X_custom)
    probs = clf.predict_proba(X_custom)

    out = pd.DataFrame({
        "text": texts,
        "predicted_label_fine": preds,
        "confidence": probs.max(axis=1)
    })
    return out

for model_name in embedding_models:
    print(f"\nCustom predictions - {model_name}")
    display(predict_custom_examples(model_name, custom_examples))

## Short conclusion template

In this project, we compared two MiniLM-based embedding models on fine-grained question classification.  
We evaluated each model using accuracy, macro F1, log loss, confusion matrices, and per-class comparisons.  
We then formed a hypothesis about weak classes, added targeted examples from the holdout set, retrained the model, and measured whether performance changed as expected.  
The final comparison should balance both **classification quality** and **runtime cost** when deciding which embedding is better for Chat Plus.